----------------- Research Part B  -  Oxford dataset ---------------------------

------------- Data Cleaning with 7day rolling avearge dataset -------------------------

In [19]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
oxcgrt = pd.read_csv('oxcgrt__7days_rolling_cases_deaths.csv') 

yougov = pd.read_csv('cleaned_data_yougov.csv') 

In [21]:
oxcgrt.columns

oxcgrt = oxcgrt.drop(columns=["ConfirmedCases","ConfirmedDeaths"])

oxcgrt.columns

Index(['RegionName', 'Date', 'H6M_Facial Coverings', 'cases_daily',
       '7days_rolling_cases', 'deaths_daily', '7days_rolling_deaths'],
      dtype='str')

In [22]:
oxcgrt.isna().sum() 

RegionName              0
Date                    0
H6M_Facial Coverings    0
cases_daily             0
7days_rolling_cases     0
deaths_daily            0
7days_rolling_deaths    0
dtype: int64

In [23]:
yougov.isna().sum()

RecordNo                                 0
Date                                     0
i2_health                                0
i9_health                                0
i11_health                               0
age                                      0
gender                                   0
state                                    0
household_size                           0
employment_status                        0
WCRex2                                   0
cantril_ladder                           0
PHQ4_1                                5943
PHQ4_2                                5957
PHQ4_3                                5956
PHQ4_4                                5957
WCRex1                                   0
r1_1                                     0
r1_2                                     0
face_mask_scale                          0
face_mask_binary                         0
general_protective_behavior_scale        0
general_protective_behavior_binary       0
protective_

In [24]:
oxcgrt.describe()

,H6M_Facial Coverings,cases_daily,7days_rolling_cases,deaths_daily,7days_rolling_deaths
count,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,1.601047,1294.384535,1290.947715,1.982791,1.975332
std,1.077753,4038.576125,3544.562956,7.391636,4.948294
min,0.000000,-53304.000000,-965.428571,-42.000000,-3.428571
25%,1.000000,0.000000,0.285714,0.000000,0.000000
50%,2.000000,1.000000,3.285714,0.000000,0.000000
75%,2.000000,308.250000,706.142857,0.000000,0.857143
max,4.000000,92264.000000,47304.428571,344.000000,58.714286


In [25]:
print(len(yougov))
print(len(oxcgrt))


39890
8600


14 day average face mask policy

In [26]:
# 14 day rolling average by state for fase mask policy 

oxcgrt["rolling_mask_policy"] = (oxcgrt.groupby("RegionName")["H6M_Facial Coverings"].rolling(window=14, min_periods=14)
                                 .mean().reset_index(level=0, drop=True)) # Rolling after groupby creates a multi-index.
                                   #This removes extra grouped index so results align back with original dataframe rows.

# # Keep days where mandate active
mandate_days = oxcgrt[oxcgrt["rolling_mask_policy"] >= 3] # continuous mandates were defined to be in place when the value was average is 3 or above - True / False

# First mandate date for each state
mandate_start = mandate_days.groupby("RegionName")["Date"].min().reset_index() # estimated start dates of continued mask mandates
mandate_start.columns = ["state", "mandate_start_date"]
print(mandate_start) 

# Check both date columns are datetime
yougov["Date"] = pd.to_datetime(yougov["Date"])
mandate_start["mandate_start_date"] = pd.to_datetime(mandate_start["mandate_start_date"])


                          state mandate_start_date
0  Australian Capital Territory         2021-08-18
1               New South Wales         2021-07-09
2            Northern Territory         2021-11-21
3                    Queensland         2021-01-17
4               South Australia         2021-07-26
5                      Tasmania         2021-10-21
6                      Victoria         2020-07-21
7             Western Australia         2021-02-08


Prepare cases and deaths to merge with yougov

In [27]:
# Prepare cases and deaths to merge with yougov

# cases and deaths 

covid_cols = oxcgrt[["RegionName","Date","cases_daily","7days_rolling_cases",
                     "deaths_daily","7days_rolling_deaths"]].rename(columns={"RegionName": "state"}) # rename the regions column

covid_cols["Date"] = pd.to_datetime(covid_cols["Date"])

Merge yougov and oxcgrt datasets

In [28]:
# Merge yougov and oxcgrt datasets by state

yougov = yougov.merge(covid_cols, on=["state", "Date"],how="left")

yougov = yougov.merge(mandate_start, on="state", how="left")

# create mandate_period binary column
#yougov["mandate_period"] = (yougov["Date"] >= yougov["mandate_start_date"]).astype(int)

yougov["mandate_period"] = (yougov["mandate_start_date"].notna() & (yougov["Date"] >= yougov["mandate_start_date"])).astype(int)


In [29]:
yougov.isna().sum()

RecordNo                                 0
Date                                     0
i2_health                                0
i9_health                                0
i11_health                               0
age                                      0
gender                                   0
state                                    0
household_size                           0
employment_status                        0
WCRex2                                   0
cantril_ladder                           0
PHQ4_1                                5943
PHQ4_2                                5957
PHQ4_3                                5956
PHQ4_4                                5957
WCRex1                                   0
r1_1                                     0
r1_2                                     0
face_mask_scale                          0
face_mask_binary                         0
general_protective_behavior_scale        0
general_protective_behavior_binary       0
protective_

Check the missing values in yougov.

In [30]:
# This method doesnt work since nas are already in nas not as NA or N/A

# print(f"comorbidities:",yougov['comorbidities'].value_counts(dropna=False)) # here NA s have been converted to nas.

# print(f"\nPHQ4_2:",yougov['PHQ4_2'].value_counts(dropna=False)) # N/A have been converted to nas.
# # print((yougov["comorbidities"]=="N/A").sum())

# # check the data type
# print(yougov["comorbidities"].dtypes)
# print(yougov["PHQ4_1"].dtypes)


# # Turning NAs and N/As into "consent_removed" category
# yougov["comorbidities"] = yougov["comorbidities"].replace("NA", "consent_removed")

# for col in yougov.columns:
#     if col.startswith("PHQ4_"):
#         yougov[col]=yougov[col].replace("N/A", "consent_removed")



In [31]:
print(yougov.isna().sum())

RecordNo                                 0
Date                                     0
i2_health                                0
i9_health                                0
i11_health                               0
age                                      0
gender                                   0
state                                    0
household_size                           0
employment_status                        0
WCRex2                                   0
cantril_ladder                           0
PHQ4_1                                5943
PHQ4_2                                5957
PHQ4_3                                5956
PHQ4_4                                5957
WCRex1                                   0
r1_1                                     0
r1_2                                     0
face_mask_scale                          0
face_mask_binary                         0
general_protective_behavior_scale        0
general_protective_behavior_binary       0
protective_

In [32]:
print(f"unique values of comorbidities:",yougov["comorbidities"].unique())

phq_cols = [col for col in yougov.columns if col.startswith("PHQ4_")] # PHQ4 columns 

for col in phq_cols:
    print(col)
    print(yougov[col].unique())

# All have been converted to nan so no use of converting NA to consent_removed. Because that string(NA) doesnt exists.
# So, lets check the difference of the na counts after converting nas into consent_removed to check whether
#  are these really missing data from consent issue.

yougov["comorbidities"] = yougov["comorbidities"].fillna("consent_removed")
yougov[phq_cols]=yougov[phq_cols].fillna("consent_removed")


# check missing value count now 

print(f"\nmissing values of comorbidities: ",yougov["comorbidities"].isna().sum())

print(f"\nnew value counts of comobidities: ",yougov["comorbidities"].value_counts(dropna=False))

for col in phq_cols:
    print(col)
    print(f"\nnew value counts of {col}:", yougov[col].value_counts(dropna=False))

unique values of comorbidities: <StringArray>
['Yes', 'No', 'Prefer_not_to_say', nan]
Length: 4, dtype: str
PHQ4_1
<StringArray>
[       'Nearly every day', 'More than half the days',
              'Not at all',            'Several days',
       'Prefer not to say',                       nan]
Length: 6, dtype: str
PHQ4_2
<StringArray>
[             'Not at all', 'More than half the days',
            'Several days',        'Nearly every day',
       'Prefer not to say',                       nan]
Length: 6, dtype: str
PHQ4_3
<StringArray>
[           'Several days', 'More than half the days',
              'Not at all',        'Nearly every day',
       'Prefer not to say',                       nan]
Length: 6, dtype: str
PHQ4_4
<StringArray>
['More than half the days',        'Nearly every day',
              'Not at all',            'Several days',
       'Prefer not to say',                       nan]
Length: 6, dtype: str

missing values of comorbidities:  0

new value counts of co

In [33]:
yougov.isna().sum()

RecordNo                              0
Date                                  0
i2_health                             0
i9_health                             0
i11_health                            0
age                                   0
gender                                0
state                                 0
household_size                        0
employment_status                     0
WCRex2                                0
cantril_ladder                        0
PHQ4_1                                0
PHQ4_2                                0
PHQ4_3                                0
PHQ4_4                                0
WCRex1                                0
r1_1                                  0
r1_2                                  0
face_mask_scale                       0
face_mask_binary                      0
general_protective_behavior_scale     0
general_protective_behavior_binary    0
protective_behavior_nomask_scale      0
comorbidities                         0


In [34]:
yougov

,RecordNo,Date,i2_health,i9_health,i11_health,age,gender,state,household_size,employment_status,...,general_protective_behavior_binary,protective_behavior_nomask_scale,comorbidities,week,cases_daily,7days_rolling_cases,deaths_daily,7days_rolling_deaths,mandate_start_date,mandate_period
0,9023,2020-06-24,0.0,Not sure,Not sure,31,Male,Western Australia,1.0,Full time employment,...,0,3.0,Yes,1,1.0,0.714286,0.0,0.000000,2021-02-08,0
1,9024,2020-06-24,2.0,No,Very willing,36,Male,Victoria,4.0,Full time employment,...,0,3.0,No,1,33.0,19.571429,0.0,0.142857,2020-07-21,0
2,9025,2020-06-24,6.0,Yes,Very willing,73,Male,Northern Territory,2.0,Retired,...,1,5.0,Yes,1,0.0,0.000000,0.0,0.000000,2021-11-21,0
3,9026,2020-06-24,20.0,Yes,Somewhat willing,58,Male,Queensland,2.0,Not working,...,0,1.0,Yes,1,0.0,0.000000,0.0,0.000000,2021-01-17,0
4,9027,2020-06-24,0.0,Yes,Very willing,65,Male,Victoria,1.0,Full time employment,...,0,1.0,No,1,33.0,19.571429,0.0,0.142857,2020-07-21,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39885,51826,2022-02-26,6.0,No,Somewhat willing,28,Male,New South Wales,5.0,Full time employment,...,1,4.0,No,44,5984.0,7298.714286,7.0,9.000000,2021-07-09,1
39886,51827,2022-02-26,4.0,Yes,Neither willing nor unwilling,32,Male,New South Wales,6.0,Unemployed,...,0,3.0,No,44,5984.0,7298.714286,7.0,9.000000,2021-07-09,1
39887,51828,2022-03-01,3.0,Yes,Very willing,31,Male,Australian Capital Territory,1.0,Full time employment,...,1,5.0,No,44,1030.0,664.714286,0.0,0.142857,2021-08-18,1
39888,51829,2022-03-01,10.0,Yes,Somewhat willing,27,Male,New South Wales,4.0,Part time employment,...,0,2.0,Yes,44,10616.0,7701.857143,5.0,8.000000,2021-07-09,1


In [35]:
yougov.to_csv("yougov_oxcgrt_rolling_merged.csv",index=False)